Instalar pysentimie nto (incluye pytorch)

In [1]:
%pip install pysentimiento

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/608.3 kB ? eta -:--:--
   ---------------------------------------- 608.3/608.3 kB 2.3 MB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 857.0/857.0 kB 3.9 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   --------------------- ------------------ 2.1/3.9 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 9.6 MB/s  0:00:00
   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------------- 610.9/610.9 kB 6.8 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ----------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

conexión con el usuario nlp_resenas

In [2]:
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from pymongo import MongoClient

RAIZ = Path.cwd().parent
load_dotenv(RAIZ / ".env")

uri = os.getenv("MONGO_URI_NLP")
if not uri:
    raise RuntimeError("No se encontró MONGO_URI_NLP en el archivo .env")

cliente = MongoClient(uri, serverSelectionTimeoutMS=5000)
db = cliente[os.getenv("MONGO_DB", "resenas_nlp")]
cliente.admin.command("ping")
print("Conectado a:", db.name)
print("Reseñas con texto por analizar:", db.resenas.count_documents({"comentario": {"$ne": None}}))
print("Temas en el catálogo:", db.temas.count_documents({"activo": True}))

Conectado a: resenas_nlp
Reseñas con texto por analizar: 108
Temas en el catálogo: 7


Cargar el modelo de sentimiento / Registra en modelos qué versión se usó, para tener trazabilidad.

In [4]:
from importlib.metadata import version
from pysentimiento import create_analyzer

VERSION_PYSENTIMIENTO = version("pysentimiento")
analizador = create_analyzer(task="sentiment", lang="es")

ETIQUETAS = {"POS": "positivo", "NEU": "neutral", "NEG": "negativo"}

def sentimiento(texto):
    """Devuelve (sentimiento, probabilidad del sentimiento, probabilidades de los tres)."""
    r = analizador.predict(texto)
    probas = {ETIQUETAS[k]: round(float(v), 4) for k, v in r.probas.items()}
    etiqueta = ETIQUETAS[r.output]
    return etiqueta, probas[etiqueta], probas

# Registrar el modelo usado (queda trazabilidad de qué versión hizo cada análisis)
db.modelos.update_one(
    {"nombre": "pysentimiento-sentiment-es", "version": VERSION_PYSENTIMIENTO},
    {"$setOnInsert": {
        "libreria": "pysentimiento",
        "parametros": {"task": "sentiment", "lang": "es", "base": "robertuito-sentiment-analysis"},
        "precision": None,
        "fecha": datetime.now().astimezone(),
        "activo": True,
    }},
    upsert=True,
)
MODELO_ID = db.modelos.find_one({"nombre": "pysentimiento-sentiment-es", "version": VERSION_PYSENTIMIENTO})["_id"]
print("Modelo registrado:", VERSION_PYSENTIMIENTO, MODELO_ID)

# Prueba
for t in ["La comida deliciosa y excelente atención", "El servicio fue lento", "Bueno"]:
    print(f"{t:45} -> {sentimiento(t)[:2]}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 557.37it/s]


Modelo registrado: 0.7.3 6b0494f4d9baec43913c4721
La comida deliciosa y excelente atención      -> ('positivo', 0.9768)
El servicio fue lento                         -> ('negativo', 0.9167)
Bueno                                         -> ('neutral', 0.6093)


Leer el catálogo de temas y palabras clave

In [5]:
import re
import unicodedata

def normalizar(texto):
    """Minúsculas y sin tildes, para comparar con las palabras clave."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))

def patron(termino):
    """'rapid*' -> palabras que empiezan con 'rapid'; 'rico' -> solo la palabra 'rico'."""
    if termino.endswith("*"):
        return re.compile(r"\b" + re.escape(termino[:-1]) + r"\w*")
    return re.compile(r"\b" + re.escape(termino) + r"\b")

temas = {t["_id"]: t["nombre"] for t in db.temas.find({"activo": True})}
PATRONES = {}                                    # tema_id -> lista de expresiones
for p in db.palabras_clave.find():
    if p["tema_id"] in temas:
        PATRONES.setdefault(p["tema_id"], []).append(patron(p["termino"]))

def dividir_fragmentos(texto):
    """Separa el comentario en frases cortas: corta en puntos, comas y en palabras como
    'pero', 'aunque' o 'sin embargo', para que 'la comida buena, pero el servicio lento'
    se analice en dos partes y cada tema reciba su propio sentimiento."""
    partes = re.split(r"[.!?;,\n]+|\b(?:pero|aunque|sin embargo|but)\b", texto, flags=re.IGNORECASE)
    return [p.strip(" ,") for p in partes if p and len(p.strip(" ,")) > 2]

for tid, pats in PATRONES.items():
    print(f"{temas[tid]:32} {len(pats):2} palabras clave")

Rapidez y tiempo de espera        9 palabras clave
Atención y trato del personal    15 palabras clave
Calidad de la comida             20 palabras clave
Precio                           10 palabras clave
Porciones y cantidad              4 palabras clave
Ambiente e instalaciones         12 palabras clave
Calidad del servicio o trabajo    9 palabras clave


Analizar cada reseña

In [6]:
from collections import Counter

ORDEN = {"negativo": 0, "neutral": 1, "positivo": 2}
resenas = list(db.resenas.find({"comentario": {"$ne": None}}))
ahora = datetime.now().astimezone()
conteo_temas, conteo_problemas = Counter(), Counter()

for i, r in enumerate(resenas, 1):
    texto = r["comentario"]

    # 1) Sentimiento de la reseña completa
    etiqueta, puntaje, probas = sentimiento(texto)
    db.analisis_sentimiento.insert_one({
        "resena_id": r["_id"], "sentimiento": etiqueta, "puntaje": puntaje,
        "probabilidades": probas, "modelo_id": MODELO_ID, "fecha": ahora,
    })

    # 2) Temas: buscar palabras clave en cada fragmento y medir el sentimiento de ese fragmento
    temas_resena, problemas_resena = [], []
    fragmentos = dividir_fragmentos(texto) or [texto]
    for tid, pats in PATRONES.items():
        encontrados = [f for f in fragmentos if any(p.search(normalizar(f)) for p in pats)]
        if not encontrados:
            continue
        evaluados = [(f, *sentimiento(f)[:2]) for f in encontrados]
        frag, sent_frag, conf = min(evaluados, key=lambda x: (ORDEN[x[1]], -x[2]))  # el más negativo
        db.temas_detectados.update_one(
            {"resena_id": r["_id"], "tema_id": tid, "metodo": "palabras_clave"},
            {"$set": {"fragmento": frag[:300], "sentimiento": sent_frag, "confianza": conf,
                      "modelo_id": MODELO_ID, "fecha": ahora}},
            upsert=True,
        )
        temas_resena.append(temas[tid]); conteo_temas[temas[tid]] += 1
        if sent_frag == "negativo":
            problemas_resena.append(temas[tid]); conteo_problemas[temas[tid]] += 1

    # 3) Resumen incrustado en la reseña (para consultas rápidas desde la API)
    db.resenas.update_one({"_id": r["_id"]}, {"$set": {"nlp": {
        "sentimiento": etiqueta, "puntaje": puntaje, "temas": temas_resena,
        "problemas": problemas_resena, "modelo_id": MODELO_ID, "analizado_en": ahora,
    }}})
    if i % 20 == 0 or i == len(resenas):
        print(f"  {i}/{len(resenas)} reseñas analizadas")

print("\nTemas mencionados:", dict(conteo_temas.most_common()))
print("Problemas detectados:", dict(conteo_problemas.most_common()))

  20/108 reseñas analizadas
  40/108 reseñas analizadas
  60/108 reseñas analizadas
  80/108 reseñas analizadas
  100/108 reseñas analizadas
  108/108 reseñas analizadas

Temas mencionados: {'Calidad de la comida': 63, 'Calidad del servicio o trabajo': 37, 'Atención y trato del personal': 36, 'Rapidez y tiempo de espera': 17, 'Precio': 16, 'Ambiente e instalaciones': 13, 'Porciones y cantidad': 6}
Problemas detectados: {'Calidad de la comida': 6, 'Calidad del servicio o trabajo': 4, 'Atención y trato del personal': 4, 'Precio': 4, 'Rapidez y tiempo de espera': 2, 'Ambiente e instalaciones': 1, 'Porciones y cantidad': 1}


Alertas de reseñas negativas

In [7]:
nuevas = 0
negativas = db.resenas.find({"$or": [{"nlp.sentimiento": "negativo"}, {"estrellas": {"$lte": 2}}]})
for r in negativas:
    res = db.alertas.update_one(
        {"resena_id": r["_id"], "tipo": "resena_negativa"},
        {"$setOnInsert": {
            "negocio_id": r["negocio_id"],
            "estado": "pendiente",
            "mensaje": f"Reseña de {r['estrellas']} estrella(s): " + (r.get("comentario") or "(sin texto)")[:150],
            "creada_en": ahora,
            "atendida_en": None,
            "atendida_por": None,
        }},
        upsert=True,
    )
    nuevas += 1 if res.upserted_id else 0
print("Alertas nuevas:", nuevas, "| Total pendientes:", db.alertas.count_documents({"estado": "pendiente"}))

Alertas nuevas: 8 | Total pendientes: 8


Reporte por negocio / El "sentimiento promedio" va de -1 (muy negativo) a 1 (muy positivo).

In [8]:
for n in db.negocios.find():
    todas = list(db.resenas.find({"negocio_id": n["_id"]}, {"fecha_creacion": 1, "nlp": 1}))
    if not todas:
        continue
    analizadas = [r for r in todas if r.get("nlp")]
    distribucion = Counter(r["nlp"]["sentimiento"] for r in analizadas)

    ids = [r["_id"] for r in analizadas]
    ultimos = list(db.analisis_sentimiento.aggregate([
        {"$match": {"resena_id": {"$in": ids}, "modelo_id": MODELO_ID}},
        {"$sort": {"fecha": -1}},
        {"$group": {"_id": "$resena_id", "p": {"$first": "$probabilidades"}}},
    ]))
    promedio = (round(sum(u["p"]["positivo"] - u["p"]["negativo"] for u in ultimos) / len(ultimos), 3)
                if ultimos else None)                      # -1 = muy negativo ... 1 = muy positivo

    problemas = list(db.temas_detectados.aggregate([
        {"$match": {"resena_id": {"$in": ids}, "sentimiento": "negativo"}},
        {"$group": {"_id": "$tema_id", "cantidad": {"$sum": 1}}},
        {"$sort": {"cantidad": -1}},
    ]))

    db.reportes.insert_one({
        "negocio_id": n["_id"],
        "periodo_inicio": min(r["fecha_creacion"] for r in todas),
        "periodo_fin": max(r["fecha_creacion"] for r in todas),
        "generado_en": ahora,
        "total_resenas": len(todas),
        "total_analizadas": len(analizadas),
        "sentimiento_promedio": promedio,
        "distribucion_sentimiento": {k: distribucion.get(k, 0) for k in ("positivo", "neutral", "negativo")},
        "principales_problemas": [{"tema_id": p["_id"], "tema": temas[p["_id"]], "cantidad": p["cantidad"]}
                                  for p in problemas],
    })

    print(f"\n{n['nombre']}")
    print(f"  Reseñas: {len(todas)} | analizadas: {len(analizadas)} | sentimiento promedio: {promedio}")
    print(f"  Distribución: {dict(distribucion)}")
    print("  Principales problemas:", ", ".join(f"{temas[p['_id']]} ({p['cantidad']})" for p in problemas) or "ninguno")


Z Express CR
  Reseñas: 44 | analizadas: 23 | sentimiento promedio: 0.693
  Distribución: {'positivo': 20, 'negativo': 1, 'neutral': 2}
  Principales problemas: Calidad del servicio o trabajo (2), Precio (1), Atención y trato del personal (1)

Marisquería La Casa del Camarón
  Reseñas: 151 | analizadas: 76 | sentimiento promedio: 0.746
  Distribución: {'positivo': 66, 'negativo': 3, 'neutral': 7}
  Principales problemas: Calidad de la comida (6), Atención y trato del personal (3), Precio (3), Rapidez y tiempo de espera (2), Calidad del servicio o trabajo (2), Ambiente e instalaciones (1), Porciones y cantidad (1)

ASM Construcción y Remodelación
  Reseñas: 9 | analizadas: 8 | sentimiento promedio: 0.832
  Distribución: {'positivo': 7, 'neutral': 1}
  Principales problemas: ninguno

ASEMINSA
  Reseñas: 1 | analizadas: 1 | sentimiento promedio: 0.652
  Distribución: {'positivo': 1}
  Principales problemas: ninguno


Ver los problemas con su recomendación

In [ ]:
recs = {x["tema_id"]: x for x in db.recomendaciones.find()}
for r in db.resenas.find({"nlp.problemas.0": {"$exists": True}}).sort("estrellas", 1):
    print(f"\n★{r['estrellas']}  {r['comentario'][:120]}")
    for td in db.temas_detectados.find({"resena_id": r["_id"], "sentimiento": "negativo"}):
        rec = recs.get(td["tema_id"])
        print(f"   ✗ {temas[td['tema_id']]}: \"{td['fragmento'][:80]}\"")
        if rec:
            print(f"     → Recomendación ({rec['prioridad']}): {rec['accion']}")


★1  Demasiado poco profesionales, el muchacho que hace el express me llamó y prácticamente me gritó y me trató super mal, mu
   ✗ Atención y trato del personal: "el muchacho que hace el express me llamó y prácticamente me gritó y me trató sup"
     → Recomendación (alta): Capacitar al personal en servicio al cliente y dar seguimiento a las quejas de trato.
   ✗ Precio: "además el pedido mal echo y súper caro en comparación con otros express que he u"
     → Recomendación (media): Comparar precios con la competencia y comunicar mejor lo que incluye cada servicio.
   ✗ Calidad del servicio o trabajo: "además el pedido mal echo y súper caro en comparación con otros express que he u"
     → Recomendación (media): Verificar el pedido o trabajo antes de entregarlo y dar seguimiento al cliente.

★3  Los ceviches son muy ricos, tiene platillos de mariscos y pescado variados a precios razonables, la baja calificación es
   ✗ Atención y trato del personal: "la baja calificación es debido a una 

: 